# Subqueries and CTEs

## Quick notes
### What this covers
- Subqueries: a SELECT inside another statement (scalar, row, table; in WHERE, FROM, SELECT).
- Correlated vs uncorrelated subqueries, `IN` vs `EXISTS`.
- CTEs (`WITH`): naming a subquery, chaining several, MATERIALIZED hints.
- Recursive CTEs: counters, date series, hierarchies, graphs, Fibonacci/factorials.
### Key points
- A scalar subquery returns one value, a table subquery returns rows, a row subquery returns one row of several columns.
- Uncorrelated = runs on its own. Correlated = mentions the outer row, so it is evaluated per outer row.
- A CTE is a named subquery that lives for one statement. Chain them with commas: `WITH a AS (...), b AS (...)`.
- A recursive CTE has an anchor part, `UNION [ALL]`, then a recursive part that reads the CTE itself.
### Gotchas (the part worth revising twice)
- `NOT IN` with a NULL anywhere in the list returns NOTHING. Use `NOT EXISTS`.
- A scalar subquery that returns 2+ rows does not error in SQLite: it silently takes the first row.
- A scalar subquery that returns 0 rows gives NULL.
- `UNION` in a recursive CTE removes duplicates, which stops cycles. `UNION ALL` on a cyclic graph runs forever unless you guard it.
- SQLite has no recursion depth limit. A runaway recursive CTE needs a `WHERE` or `LIMIT` to stop.
- A CTE is not a table. It is gone after the statement ends.
### When it matters
- "Above average", "no orders", "latest per group", org charts, category trees, route finding, calendars with no gaps.
### One-line recap example
```sql
WITH avg_sal AS (SELECT AVG(salary) AS a FROM employees)
SELECT first_name FROM employees, avg_sal WHERE salary > a;
```

## 01. Seed data
- Employees with a manager tree, departments, and orders.
- Small on purpose so every result can be checked by eye.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE employees (
    emp_id INTEGER PRIMARY KEY, first_name TEXT NOT NULL, dept TEXT NOT NULL,
    manager_id INTEGER REFERENCES employees(emp_id), salary REAL NOT NULL
);
INSERT INTO employees VALUES
 (1,'Alice','Engineering',NULL,95000),
 (2,'Bob','Engineering',1,72000),
 (3,'Carla','Sales',1,68000),
 (4,'Dev','Sales',3,61000),
 (5,'Eve','Sales',3,58000),
 (6,'Finn','Marketing',1,75000),
 (7,'Gus','Engineering',2,70000);
CREATE TABLE customers (cust_id INTEGER PRIMARY KEY, name TEXT NOT NULL);
CREATE TABLE orders (order_id INTEGER PRIMARY KEY, cust_id INTEGER, amount REAL NOT NULL, order_date TEXT NOT NULL);
INSERT INTO customers VALUES (1,'Ann'),(2,'Ben'),(3,'Cy'),(4,'Di');
INSERT INTO orders VALUES
 (1,1,120,'2024-01-05'),(2,1,80,'2024-02-10'),(3,2,300,'2024-01-20'),
 (4,3,50,'2024-03-01'),(5,NULL,999,'2024-03-02');
SELECT first_name, dept, manager_id, salary FROM employees ORDER BY emp_id;

first_name | dept        | manager_id | salary 
-----------+-------------+------------+--------
Alice      | Engineering | NULL       | 95000.0
Bob        | Engineering | 1          | 72000.0
Carla      | Sales       | 1          | 68000.0
Dev        | Sales       | 3          | 61000.0
Eve        | Sales       | 3          | 58000.0
Finn       | Marketing   | 1          | 75000.0
Gus        | Engineering | 2          | 70000.0
(7 rows)



## 02. Scalar subqueries
- Returns exactly one row and one column, so it can be used wherever a single value goes.
- Uncorrelated: it runs once and the value is reused.
- Handy for "compare each row to an overall number".

In [3]:
%%sql
SELECT first_name, salary,
       (SELECT ROUND(AVG(salary)) FROM employees) AS company_avg,
       salary - (SELECT ROUND(AVG(salary)) FROM employees) AS diff
FROM employees
ORDER BY salary DESC;

first_name | salary  | company_avg | diff    
-----------+---------+-------------+---------
Alice      | 95000.0 | 71286.0     | 23714.0 
Finn       | 75000.0 | 71286.0     | 3714.0  
Bob        | 72000.0 | 71286.0     | 714.0   
Gus        | 70000.0 | 71286.0     | -1286.0 
Carla      | 68000.0 | 71286.0     | -3286.0 
Dev        | 61000.0 | 71286.0     | -10286.0
Eve        | 58000.0 | 71286.0     | -13286.0
(7 rows)



## 03. Scalar subquery gotchas
- 0 rows gives NULL, not an error.
- 2+ rows: SQLite quietly uses the first row. Other databases raise an error. Do not rely on it.
- Make the subquery truly single-valued (aggregate, or `LIMIT 1` with an `ORDER BY`).

In [4]:
%%sql
SELECT (SELECT salary FROM employees WHERE dept = 'Nowhere') AS zero_rows,
       (SELECT salary FROM employees WHERE dept = 'Sales' ORDER BY salary) AS many_rows_takes_first,
       (SELECT MAX(salary) FROM employees WHERE dept = 'Sales') AS safe_max;

zero_rows | many_rows_takes_first | safe_max
----------+-----------------------+---------
NULL      | 58000.0               | 68000.0 
(1 row)



## 04. Table subqueries with IN
- Returns one column of many rows. `x IN (subquery)` keeps rows whose value appears in it.
- Reads like "matches any of these".

In [5]:
%%sql
SELECT first_name, dept
FROM employees
WHERE emp_id IN (SELECT manager_id FROM employees WHERE manager_id IS NOT NULL)
ORDER BY first_name;

first_name | dept       
-----------+------------
Alice      | Engineering
Bob        | Engineering
Carla      | Sales      
(3 rows)



## 05. Row subqueries
- A subquery returning one row with several columns, compared with a row value `(a, b) = (...)`.
- Good for "the row that has the top salary in its department".

In [6]:
%%sql
SELECT first_name, dept, salary
FROM employees
WHERE (dept, salary) IN (SELECT dept, MAX(salary) FROM employees GROUP BY dept)
ORDER BY dept;

first_name | dept        | salary 
-----------+-------------+--------
Alice      | Engineering | 95000.0
Finn       | Marketing   | 75000.0
Carla      | Sales       | 68000.0
(3 rows)



## 06. Correlated subqueries
- The inner query uses a column of the outer row, so it re-runs for each outer row.
- Classic use: "earns more than the average of their own department".
- Can be slow on big tables without an index on the correlated column (see the query-plans topic, EXPLAIN QUERY PLAN).

In [7]:
%%sql
SELECT e.first_name, e.dept, e.salary
FROM employees e
WHERE e.salary > (SELECT AVG(x.salary) FROM employees x WHERE x.dept = e.dept)
ORDER BY e.dept, e.first_name;

first_name | dept        | salary 
-----------+-------------+--------
Alice      | Engineering | 95000.0
Carla      | Sales       | 68000.0
(2 rows)



## 07. Correlated subquery in SELECT
- Computes one extra value per row, like a mini lookup.
- Same answer as a join + GROUP BY, but a LEFT JOIN is often faster and clearer (see 02-Querying/01-Joins).

In [8]:
%%sql
SELECT c.name,
       (SELECT COUNT(*) FROM orders o WHERE o.cust_id = c.cust_id) AS n_orders,
       (SELECT COALESCE(SUM(amount),0) FROM orders o WHERE o.cust_id = c.cust_id) AS total
FROM customers c
ORDER BY c.cust_id;

name | n_orders | total
-----+----------+------
Ann  | 2        | 200.0
Ben  | 1        | 300.0
Cy   | 1        | 50.0 
Di   | 0        | 0    
(4 rows)



## 08. EXISTS vs IN
- `EXISTS` asks "is there at least one row?". It stops at the first match and ignores what the subquery selects.
- `IN` compares against a list of values.
- For positive checks they give the same rows. The difference shows up with NULLs (next section).

In [9]:
%%sql
SELECT name FROM customers c
WHERE EXISTS (SELECT 1 FROM orders o WHERE o.cust_id = c.cust_id)
ORDER BY name;
SELECT name FROM customers
WHERE cust_id IN (SELECT cust_id FROM orders)
ORDER BY name;

name
----
Ann 
Ben 
Cy  
(3 rows)

name
----
Ann 
Ben 
Cy  
(3 rows)



## 09. NOT IN vs NOT EXISTS (the NULL trap)
- `orders` has a row with `cust_id` NULL (an order with no customer).
- `NOT IN` against a list containing NULL is never true, so you get zero rows.
- `NOT EXISTS` and `LEFT JOIN ... IS NULL` handle it correctly.

In [10]:
%%sql
-- Bad: returns nothing, even though Di has no orders
SELECT name FROM customers WHERE cust_id NOT IN (SELECT cust_id FROM orders) ORDER BY name;
-- Fix 1: filter the NULLs out
SELECT name FROM customers WHERE cust_id NOT IN (SELECT cust_id FROM orders WHERE cust_id IS NOT NULL) ORDER BY name;
-- Fix 2: NOT EXISTS (preferred)
SELECT name FROM customers c WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.cust_id = c.cust_id) ORDER BY name;

name
----
(0 rows)

name
----
Di  
(1 row)

name
----
Di  
(1 row)



## 10. Subquery in FROM (derived table)
- A subquery in `FROM` acts like a temporary table and needs an alias.
- Use it to aggregate first, then filter or aggregate again.

In [11]:
%%sql
SELECT dept, ROUND(AVG(total_pay)) AS avg_of_dept_totals
FROM (SELECT dept, manager_id, SUM(salary) AS total_pay FROM employees GROUP BY dept, manager_id) AS t
GROUP BY dept
ORDER BY dept;

dept        | avg_of_dept_totals
------------+-------------------
Engineering | 79000.0           
Marketing   | 75000.0           
Sales       | 93500.0           
(3 rows)



## 11. ANY / ALL alternatives
- SQLite has no `ANY`/`ALL`. Use `> (SELECT MAX(...))` for "greater than all" and `> (SELECT MIN(...))` for "greater than any".

In [12]:
%%sql
SELECT first_name, salary FROM employees
WHERE salary > (SELECT MAX(salary) FROM employees WHERE dept = 'Sales')
ORDER BY salary;

first_name | salary 
-----------+--------
Gus        | 70000.0
Bob        | 72000.0
Finn       | 75000.0
Alice      | 95000.0
(4 rows)



## 12. CTEs: naming a subquery
- `WITH name AS (SELECT ...)` then use `name` like a table.
- Same result as a derived table, but reads top to bottom and can be reused.
- Bad vs good: nested subqueries vs a CTE.

In [13]:
%%sql
-- Bad: nested, hard to read
SELECT first_name FROM employees
WHERE salary > (SELECT AVG(salary) FROM employees WHERE dept IN (SELECT dept FROM employees WHERE first_name = 'Dev'))
ORDER BY first_name;
-- Good: named steps
WITH devs_dept AS (SELECT dept FROM employees WHERE first_name = 'Dev'),
     dept_avg   AS (SELECT AVG(salary) AS a FROM employees WHERE dept IN (SELECT dept FROM devs_dept))
SELECT first_name FROM employees, dept_avg WHERE salary > a ORDER BY first_name;

first_name
----------
Alice     
Bob       
Carla     
Finn      
Gus       
(5 rows)

first_name
----------
Alice     
Bob       
Carla     
Finn      
Gus       
(5 rows)



## 13. Chaining multiple CTEs
- Separate CTEs with commas, one `WITH`. Later CTEs can use earlier ones.
- Each step does one small job: easy to test by selecting from it alone.

In [14]:
%%sql
WITH per_customer AS (
    SELECT cust_id, SUM(amount) AS total FROM orders WHERE cust_id IS NOT NULL GROUP BY cust_id
),
ranked AS (
    SELECT cust_id, total, total * 100.0 / (SELECT SUM(total) FROM per_customer) AS pct FROM per_customer
)
SELECT c.name, r.total, ROUND(r.pct, 1) AS pct_of_total
FROM ranked r JOIN customers c USING (cust_id)
ORDER BY r.total DESC;

name | total | pct_of_total
-----+-------+-------------
Ben  | 300.0 | 54.5        
Ann  | 200.0 | 36.4        
Cy   | 50.0  | 9.1         
(3 rows)



## 14. CTE reused twice
- The same CTE can be referenced many times in the main query (self-join of a CTE).
- Writing it once avoids copy-paste bugs.

In [15]:
%%sql
WITH dept_pay AS (SELECT dept, SUM(salary) AS total FROM employees GROUP BY dept)
SELECT a.dept AS dept_a, b.dept AS dept_b, a.total - b.total AS gap
FROM dept_pay a JOIN dept_pay b ON a.total > b.total
ORDER BY a.dept, b.dept;

dept_a      | dept_b    | gap     
------------+-----------+---------
Engineering | Marketing | 162000.0
Engineering | Sales     | 50000.0 
Sales       | Marketing | 112000.0
(3 rows)



## 15. MATERIALIZED / NOT MATERIALIZED
- Hints (SQLite 3.35+): `AS MATERIALIZED (...)` computes the CTE once into a temp result. `AS NOT MATERIALIZED (...)` inlines it like a view.
- By default SQLite materializes a CTE used 2+ times and inlines one used once.
- Use MATERIALIZED when the CTE is expensive and reused. Use NOT MATERIALIZED to let filters push down into it.
- The result rows are identical; only speed changes.

In [16]:
%%sql
WITH big AS MATERIALIZED (SELECT dept, salary FROM employees WHERE salary > 60000)
SELECT dept, COUNT(*) AS n FROM big GROUP BY dept ORDER BY dept;
WITH big AS NOT MATERIALIZED (SELECT dept, salary FROM employees WHERE salary > 60000)
SELECT dept, COUNT(*) AS n FROM big GROUP BY dept ORDER BY dept;

dept        | n
------------+--
Engineering | 3
Marketing   | 1
Sales       | 2
(3 rows)

dept        | n
------------+--
Engineering | 3
Marketing   | 1
Sales       | 2
(3 rows)



## 16. Recursive CTE: the shape
- Anchor query (starting rows) `UNION ALL` recursive query (reads the CTE name itself).
- It repeats until the recursive part returns no new rows.
- Always include a stopping condition (`WHERE n < 10`).

In [17]:
%%sql
WITH RECURSIVE counter(n) AS (
    SELECT 1
    UNION ALL
    SELECT n + 1 FROM counter WHERE n < 10
)
SELECT n FROM counter ORDER BY n;

n 
--
1 
2 
3 
4 
5 
6 
7 
8 
9 
10
(10 rows)



## 17. Recursive CTE: Fibonacci and factorials
- Carry extra columns along to hold state between steps.
- Fibonacci needs two previous values. Factorial multiplies a running product.

In [18]:
%%sql
WITH RECURSIVE fib(n, a, b) AS (
    SELECT 1, 0, 1
    UNION ALL
    SELECT n + 1, b, a + b FROM fib WHERE n < 12
)
SELECT n, a AS fibonacci FROM fib ORDER BY n;
WITH RECURSIVE fact(n, f) AS (
    SELECT 1, 1
    UNION ALL
    SELECT n + 1, f * (n + 1) FROM fact WHERE n < 10
)
SELECT n, f AS factorial FROM fact ORDER BY n;

n  | fibonacci
---+----------
1  | 0        
2  | 1        
3  | 1        
4  | 2        
5  | 3        
6  | 5        
7  | 8        
8  | 13       
9  | 21       
10 | 34       
11 | 55       
12 | 89       
(12 rows)

n  | factorial
---+----------
1  | 1        
2  | 2        
3  | 6        
4  | 24       
5  | 120      
6  | 720      
7  | 5040     
8  | 40320    
9  | 362880   
10 | 3628800  
(10 rows)



## 18. Recursive CTE: a date series with fixed dates
- Generate every day between two dates, then LEFT JOIN data onto it to show days with no activity.
- Dates are fixed here so the output never changes.

In [19]:
%%sql
WITH RECURSIVE days(d) AS (
    SELECT '2024-03-01'
    UNION ALL
    SELECT date(d, '+1 day') FROM days WHERE d < '2024-03-05'
)
SELECT days.d, COALESCE(SUM(o.amount), 0) AS sales
FROM days LEFT JOIN orders o ON o.order_date = days.d
GROUP BY days.d
ORDER BY days.d;

d          | sales
-----------+------
2024-03-01 | 50.0 
2024-03-02 | 999.0
2024-03-03 | 0    
2024-03-04 | 0    
2024-03-05 | 0    
(5 rows)



## 19. Recursive CTE: hierarchy with depth and path
- Walk the org chart from the top. Carry `depth` and a text `path` so you can indent or sort as a tree.
- Sorting by `path` gives a natural parent-then-children order.

In [20]:
%%sql
WITH RECURSIVE org(emp_id, first_name, depth, path) AS (
    SELECT emp_id, first_name, 0, first_name FROM employees WHERE manager_id IS NULL
    UNION ALL
    SELECT e.emp_id, e.first_name, o.depth + 1, o.path || ' > ' || e.first_name
    FROM employees e JOIN org o ON e.manager_id = o.emp_id
)
SELECT substr('        ', 1, depth * 2) || first_name AS tree, depth, path
FROM org
ORDER BY path;

tree    | depth | path               
--------+-------+--------------------
Alice   | 0     | Alice              
  Bob   | 1     | Alice > Bob        
    Gus | 2     | Alice > Bob > Gus  
  Carla | 1     | Alice > Carla      
    Dev | 2     | Alice > Carla > Dev
    Eve | 2     | Alice > Carla > Eve
  Finn  | 1     | Alice > Finn       
(7 rows)



## 20. Recursive CTE: everyone under a manager
- Start the anchor at one person instead of the root to get a whole subtree.
- Aggregate on the result: how many people sit under Carla?

In [21]:
%%sql
WITH RECURSIVE team(emp_id) AS (
    SELECT emp_id FROM employees WHERE first_name = 'Carla'
    UNION ALL
    SELECT e.emp_id FROM employees e JOIN team t ON e.manager_id = t.emp_id
)
SELECT COUNT(*) - 1 AS people_under_carla FROM team;

people_under_carla
------------------
2                 
(1 row)



## 21. Graph reachability and the cycle guard (UNION vs UNION ALL)
- Edges can form a loop (A to B to C to A). With `UNION ALL` the walk never ends, because new rows keep appearing.
- Plain `UNION` drops rows already seen, so the loop ends by itself. This only works if the rows are identical, so do not carry a growing `depth` or `path` column.
- If you need the path, check `path NOT LIKE '%' || next || '%'` instead.

In [22]:
%%sql
CREATE TABLE flights (src TEXT, dst TEXT);
INSERT INTO flights VALUES ('A','B'),('B','C'),('C','A'),('C','D'),('E','F');
-- Reachable from A. UNION removes repeats so the A-B-C-A loop terminates.
WITH RECURSIVE reach(city) AS (
    SELECT 'A'
    UNION
    SELECT f.dst FROM flights f JOIN reach r ON f.src = r.city
)
SELECT city FROM reach ORDER BY city;
-- With paths, guard against revisiting a city
WITH RECURSIVE trip(city, path) AS (
    SELECT 'A', 'A'
    UNION ALL
    SELECT f.dst, t.path || '>' || f.dst
    FROM flights f JOIN trip t ON f.src = t.city
    WHERE t.path NOT LIKE '%' || f.dst || '%'
)
SELECT path FROM trip ORDER BY length(path), path;

city
----
A   
B   
C   
D   
(4 rows)

path   
-------
A      
A>B    
A>B>C  
A>B>C>D
(4 rows)



## 22. The recursion limit gotcha
- SQLite has no built-in recursion depth cap. A recursive CTE with no stop condition runs until you run out of memory or patience.
- Safety nets: a `WHERE` on the counter, `UNION` for cycles, or a `LIMIT` on the outer select (SQLite stops generating once the limit is met).
- Other databases (SQL Server, PostgreSQL settings) may cap recursion; do not depend on that.

In [23]:
%%sql
-- No natural end here, but LIMIT makes it safe
WITH RECURSIVE forever(n) AS (
    SELECT 1 UNION ALL SELECT n + 1 FROM forever
)
SELECT n FROM forever LIMIT 5;

n
-
1
2
3
4
5
(5 rows)



## 23. Choosing: subquery, CTE or join?
- One simple value or yes/no check: subquery.
- Several steps, or the same result used twice: CTE.
- Combining columns from two tables: usually a join (see 02-Querying/01-Joins).
- Tree or graph of unknown depth: recursive CTE.
- Per-group ranking, "latest per group": window functions (see 03-Window-Functions/01-Window-Functions).